# Notebook 04 — Explore with Genie in the UI

**What you’ll do:** Open your Genie space in the browser and ask questions in plain language.

**Tip:** Notebook **01** fixes the random **seed**, so the **Reference SQL** section below shows **expected numbers** you can compare to Genie’s answers.

**Before you start:** **01**, **02**, **03**. **Workshop order:** 01 → … → **04** → 05 → … → 09.


## Compute

Use **Serverless** (recommended).


## Configuration

Use the **same** **Catalog** and **Schema** as **01** / **02**. This notebook **builds** the Genie link from your workspace and the space id in `workshop_config` so the link stays valid.


In [ ]:
import html
import re

from databricks.sdk import WorkspaceClient

dbutils.widgets.text("catalog", "gm_ama_demos", "Catalog")
dbutils.widgets.text("schema", "genie_workshop_manufacturing", "Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
fqn = f"{CATALOG}.{SCHEMA}"


def genie_ui_room_url(host: str, space_id: str) -> str:
    h = (host or "").rstrip("/")
    sid = str(space_id or "").strip()
    m = re.search(r"adb-(\d+)\.", h)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{h}/genie/rooms/{sid}{q}"


w = WorkspaceClient()
_host = w.config.host.rstrip("/")

cfg = spark.sql(
    f"SELECT value, space_name FROM {fqn}.workshop_config WHERE key = 'genie_space_id'"
).collect()
if not cfg:
    print("Run notebook 02 first — no genie_space_id in workshop_config.")
else:
    _sid = cfg[0]["value"]
    _name = cfg[0]["space_name"]
    _u = genie_ui_room_url(_host, _sid)
    print("Primary space id:", _sid)
    print("Primary Genie UI URL (computed):", _u)
    displayHTML(
        "<p>"
        + html.escape(str(_name))
        + ' &mdash; <a href="'
        + html.escape(_u, quote=True)
        + '" target="_blank" rel="noopener">Open in Genie</a></p>'
    )


## Quality and OEE — try in Genie, then compare

1. **What is the average OEE by plant for 2024?**
   - **Expected answer:** One row per plant with **average OEE %** = `AVG(oee_score) * 100` over `quality_metrics_daily` rows where `YEAR(date) = 2024`, joined to `plants` for the name. Ordering may vary; values should match the reference table below.

2. **Which production lines have the highest defect rates in 2024?**
   - **Expected answer:** Defect rate **%** = `COUNT(defect_detected) / COUNT(unit_produced) * 100` on `production_events` in 2024, grouped by line (join `production_lines`). Highest rates appear at the **top** when sorted descending.

3. **Show first pass yield trend by month for 2024.**
   - **Expected answer:** One row per **month** with **avg FPY %** = `AVG(first_pass_yield) * 100` from `quality_metrics_daily` where `YEAR(date)=2024`, e.g. `DATE_TRUNC('month', date)`.


## Production and downtime — try in Genie, then compare

1. **Compare total downtime minutes across plants.**
   - **Expected answer:** **Sum** of `downtime_minutes` from `quality_metrics_daily`, grouped by plant (join `plants`).

2. **How many unit_produced events occurred in Q1 2024?**
   - **Expected answer:** A **single count** of rows in `production_events` where `event_type = 'unit_produced'` and `event_date` is in **2024-01-01** through **2024-03-31** (inclusive).


## Safety — try in Genie, then compare

1. **List critical safety incidents.**
   - **Expected answer:** Rows from `safety_incidents` where `severity = 'Critical'`, typically ordered by `incident_date` descending.

2. **How many incidents by severity?**
   - **Expected answer:** **Counts** grouped by `severity` (values include Low, Medium, High, Critical per notebook 01).


## Reference SQL — ground truth for the prompts above

Run this cell after **01_setup_data** on the **same** catalog/schema. Use the outputs as the **expected answers** when validating Genie.


In [ ]:
print("=== 1. Average OEE % by plant (2024) ===")
display(
    spark.sql(
        f"""
SELECT p.plant_name, ROUND(AVG(q.oee_score) * 100, 2) AS avg_oee_pct
FROM {fqn}.quality_metrics_daily q
JOIN {fqn}.plants p ON q.plant_id = p.plant_id
WHERE YEAR(q.date) = 2024
GROUP BY p.plant_name
ORDER BY avg_oee_pct DESC
"""
    )
)

print("=== 2. Defect rate % by production line (2024, top 15) ===")
display(
    spark.sql(
        f"""
SELECT
  pl.line_name,
  ROUND(
    SUM(CASE WHEN pe.event_type = 'defect_detected' THEN 1 ELSE 0 END) * 100.0
    / NULLIF(SUM(CASE WHEN pe.event_type = 'unit_produced' THEN 1 ELSE 0 END), 0),
    2
  ) AS defect_rate_pct
FROM {fqn}.production_events pe
JOIN {fqn}.production_lines pl ON pe.production_line_id = pl.line_id
WHERE YEAR(pe.event_date) = 2024
GROUP BY pl.line_id, pl.line_name
ORDER BY defect_rate_pct DESC
LIMIT 15
"""
    )
)

print("=== 3. First pass yield % by month (2024) ===")
display(
    spark.sql(
        f"""
SELECT
  DATE_TRUNC('month', q.date) AS month,
  ROUND(AVG(q.first_pass_yield) * 100, 2) AS avg_fpy_pct
FROM {fqn}.quality_metrics_daily q
WHERE YEAR(q.date) = 2024
GROUP BY DATE_TRUNC('month', q.date)
ORDER BY month
"""
    )
)

print("=== 4. Total downtime minutes by plant ===")
display(
    spark.sql(
        f"""
SELECT p.plant_name, ROUND(SUM(q.downtime_minutes), 2) AS total_downtime_minutes
FROM {fqn}.quality_metrics_daily q
JOIN {fqn}.plants p ON q.plant_id = p.plant_id
GROUP BY p.plant_name
ORDER BY total_downtime_minutes DESC
"""
    )
)

print("=== 5. unit_produced events in Q1 2024 ===")
display(
    spark.sql(
        f"""
SELECT CAST(COUNT(*) AS BIGINT) AS unit_produced_events_q1_2024
FROM {fqn}.production_events
WHERE event_type = 'unit_produced'
  AND event_date >= DATE('2024-01-01')
  AND event_date < DATE('2024-04-01')
"""
    )
)

print("=== 6. Critical safety incidents ===")
display(
    spark.sql(
        f"""
SELECT incident_id, incident_date, production_line_id, severity, description
FROM {fqn}.safety_incidents
WHERE severity = 'Critical'
ORDER BY incident_date DESC
"""
    )
)

print("=== 7. Incident counts by severity ===")
display(
    spark.sql(
        f"""
SELECT severity, COUNT(*) AS incident_count
FROM {fqn}.safety_incidents
GROUP BY severity
ORDER BY severity
"""
    )
)
